<a href="https://colab.research.google.com/github/amrzhd/EEG-MSCNN/blob/main/Dorna_Kahan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installing Packages

In [ ]:
!pip install accelerate -U -q

In [ ]:
!pip install git+https://github.com/huggingface/datasets.git

In [ ]:
!pip install -U bitsandbytes

In [ ]:
!pip install trl

#Libraries Used

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict

# Hugging Face
import wandb
from google.colab import userdata
from huggingface_hub import login

# Torch
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.utils.checkpoint as cp

from functools import partial
cp.checkpoint = partial(cp.checkpoint, use_reentrant=True)
cp.checkpoint_sequential = partial(cp.checkpoint_sequential, use_reentrant=True)

# Peft
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Transformers
import transformers
from transformers import (
    Trainer,
    pipeline,
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
)

from trl import (
    SFTTrainer,
    DataCollatorForCompletionOnlyLM,
)


#HF Login

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
print(HF_TOKEN)
print(WANDB_API_KEY)
login(token=HF_TOKEN)
wandb.login(key=WANDB_API_KEY)

#Building Dataset

##Persian Text Normalizer

###Mappings

In [ ]:
SUFFIX_HA_EXCEPTION_LIST = [
    'منها',
    'رها',
    'بهای',
    'بها',
    'انتهای',
    'انتها',
    'منتها',
]

SPACE_MAPPING_LIST = [
    chr(9),     # \t Tab (^9)
    chr(10),    # \n Line Feed
    chr(11),    # \v Vertical Tab
    chr(12),    # \f Form Feed
    chr(13),    # \r Carriage Return (^13)
    chr(31),    # Unit Separator (^31)
    chr(133),   # Next Line (NEL)
    chr(160),   # No-break space
    chr(172),   # Not Sign
    chr(5760),  # Ogham space mark
    chr(8192),  # En quad
    chr(8193),  # Em quad
    chr(8194),  # En space
    chr(8195),  # Em space
    chr(8196),  # Three-per-em space
    chr(8197),  # Four-per-em space
    chr(8198),  # Six-per-em space
    chr(8199),  # Figure space
    chr(8200),  # Punctuation space
    chr(8201),  # Thin space
    chr(8202),  # Hair space
    chr(8203),  # Zero-width space
    chr(8204),  # Zero-width non-joiner
    chr(8206),  # Left-to-Right Mark
    chr(8207),  # Right-to-left Mark
    chr(8232),  # Line separator
    chr(8233),  # Paragraph separator
    chr(8235),  # ⁫Right-to-left embedding
    chr(8236),  # ⁬Pop directional formatting
    chr(8239),  # Narrow no-break space
    chr(8287),  # Medium mathematical space
    chr(8301),  # Zero-width space (Active Arabic form shaping)
    chr(12288), # Ideographic space
    chr(65279), # Byte order mark / zero-width no-break space
]

ARABIC2PERSIAN_MAPPING_DICT = {
    # آ
    chr(65153): chr(1570),  # ﺁ
    chr(65154): chr(1570),  # ﺂ

    # ا
    chr(64336): chr(1575),  # ﭐ
    chr(64337): chr(1575),  # ﭑ
    chr(65165): chr(1575),  # ﺍ
    chr(65166): chr(1575),  # ﺎ

    # أ
    chr(1573):  chr(1571),  # ﺇ
    chr(65156): chr(1571),  # أ
    chr(65159): chr(1571),  # ﺇ
    chr(65160): chr(1571),  # ﺈ

    # ب
    chr(65167): chr(1576),  # ﺏ
    chr(65168): chr(1576),  # ﺐ
    chr(65169): chr(1576),  # ﺑ
    chr(65170): chr(1576),  # ﺒ

    # پ
    chr(64342): chr(1662),  # ﭖ
    chr(64343): chr(1662),  # ﭗ
    chr(64344): chr(1662),  # ﭘ
    chr(64345): chr(1662),  # ﭙ

    # ت
    chr(65173): chr(1578),  # ﺕ
    chr(65174): chr(1578),  # ﺖ
    chr(65175): chr(1578),  # ﺗ
    chr(65176): chr(1578),  # ﺘ

    # ث
    chr(65177): chr(1579),  # ﺙ
    chr(65178): chr(1579),  # ﺚ
    chr(65179): chr(1579),  # ﺛ
    chr(65180): chr(1579),  # ﺜ

    # ج
    chr(65181): chr(1580),  # ﺝ
    chr(65182): chr(1580),  # ﺞ
    chr(65183): chr(1580),  # ﺟ
    chr(65184): chr(1580),  # ﺠ

    # چ
    chr(1671):  chr(1670),  #
    chr(64378): chr(1670),  # ﭺ
    chr(64379): chr(1670),  # ﭻ
    chr(64380): chr(1670),  # ﭼ
    chr(64381): chr(1670),  # ﭽ

    # ح
    chr(65185): chr(1581),  # ﺡ
    chr(65186): chr(1581),  # ﺢ
    chr(65187): chr(1581),  # ﺣ
    chr(65188): chr(1581),  # ﺤ

    # خ
    chr(65189): chr(1582),  # ﺥ
    chr(65190): chr(1582),  # ﺦ
    chr(65191): chr(1582),  # ﺧ
    chr(65192): chr(1582),  # ﺨ

    # د
    chr(1929) : chr(1583),  # މ
    chr(65193): chr(1583),  # ﺩ
    chr(65194): chr(1583),  # ﺪ

    # ذ
    chr(65195): chr(1584),  # ﺫ
    chr(65196): chr(1584),  # ﺬ

    # ر
    chr(1883):  chr(1585),  # ݛ
    chr(1920):  chr(1585),  # ހ
    chr(65197): chr(1585),  # ﺭ
    chr(65198): chr(1585),  # ﺮ

    # ز
    chr(65199): chr(1586),  # ﺯ
    chr(65200): chr(1586),  # ﺰ

    # ژ
    chr(64394): chr(1688),  # ﮊ
    chr(64395): chr(1688),  # ﮋ

    # س
    chr(65201): chr(1587),  # ﺱ
    chr(65202): chr(1587),  # ﺲ
    chr(65203): chr(1587),  # ﺳ
    chr(65204): chr(1587),  # ﺴ

    # ش
    chr(1884):  chr(1588),  # ݜ
    chr(65205): chr(1588),  # ﺵ
    chr(65206): chr(1588),  # ﺶ
    chr(65207): chr(1588),  # ﺷ
    chr(65208): chr(1588),  # ﺸ

    # ص
    chr(65209): chr(1589),  # ﺹ
    chr(65210): chr(1589),  # ﺺ
    chr(65211): chr(1589),  # ﺻ
    chr(65212): chr(1589),  # ﺼ

    # ض
    chr(65213): chr(1590),  # ﺽ
    chr(65214): chr(1590),  # ﺾ
    chr(65215): chr(1590),  # ﺿ
    chr(65216): chr(1590),  # ﻀ

    # ط
    chr(65217): chr(1591),  # ﻁ
    chr(65218): chr(1591),  # ﻂ
    chr(65219): chr(1591),  # ﻃ
    chr(65220): chr(1591),  # ﻄ

    # ظ
    chr(65221): chr(1592),  # ﻅ
    chr(65222): chr(1592),  # ﻆ
    chr(65223): chr(1592),  # ﻇ
    chr(65224): chr(1592),  # ﻈ

    # ع
    chr(65225): chr(1593),  # ﻉ
    chr(65226): chr(1593),  # ﻊ
    chr(65227): chr(1593),  # ﻋ
    chr(65228): chr(1593),  # ﻌ

    # غ
    chr(65229): chr(1594),  # ﻍ
    chr(65230): chr(1594),  # ﻎ
    chr(65231): chr(1594),  # ﻏ
    chr(65232): chr(1594),  # ﻐ

    # ف
    chr(65233): chr(1601),  # ﻑ
    chr(65234): chr(1601),  # ﻒ
    chr(65235): chr(1601),  # ﻓ
    chr(65236): chr(1601),  # ﻔ

    # ق
    chr(65237): chr(1602),  # ﻕ
    chr(65238): chr(1602),  # ﻖ
    chr(65239): chr(1602),  # ﻗ
    chr(65240): chr(1602),  # ﻘ

    # ک
    chr(1603):  chr(1705),  # ك
    chr(1706):  chr(1705),  # ڪ
    chr(64398): chr(1705),  # ﮎ
    chr(64399): chr(1705),  # ﮏ
    chr(64400): chr(1705),  # ﮐ
    chr(64401): chr(1705),  # ﮑ
    chr(65241): chr(1705),  # ﻙ
    chr(65242): chr(1705),  # ﻚ
    chr(65243): chr(1705),  # ﻛ
    chr(65244): chr(1705),  # ﻜ

    # گ
    chr(1667):  chr(1711),  # ݣ
    chr(1679):  chr(1711),  # ݿ
    chr(1701):  chr(1711),  # ڭ
    chr(1706):  chr(1711),  # گ
    chr(64402): chr(1711),  # ﮒ
    chr(64403): chr(1711),  # ﮓ
    chr(64404): chr(1711),  # ﮔ
    chr(64405): chr(1711),  # ﮕ

    # ل
    chr(65245): chr(1604),  # ﻝ
    chr(65246): chr(1604),  # ﻞ
    chr(65247): chr(1604),  # ﻟ
    chr(65248): chr(1604),  # ﻠ

    # م
    chr(65249): chr(1605),  # ﻡ
    chr(65250): chr(1605),  # ﻢ
    chr(65251): chr(1605),  # ﻣ
    chr(65252): chr(1605),  # ﻤ

    # ن
    chr(65253): chr(1606),  # ﻥ
    chr(65254): chr(1606),  # ﻦ
    chr(65255): chr(1606),  # ﻧ
    chr(65256): chr(1606),  # ﻨ

    # و
    chr(1572):  chr(1608),  # ؤ
    chr(1700):  chr(1608),  # ڤ
    chr(1928):  chr(1608),  # ވ
    chr(65261): chr(1608),  # ﻭ
    chr(65262): chr(1608),  # ﻮ
    chr(65157): chr(1608),  # ﺅ
    chr(65158): chr(1608),  # ﺆ

    # ه
    chr(1577):  chr(1607),  # ه
    chr(1726):  chr(1607),  # ھ
    chr(1729):  chr(1607),  # ہ
    chr(1749):  chr(1607),  # ە
    chr(64426): chr(1607),  # ﮪ
    chr(64427): chr(1607),  # ﮫ
    chr(64428): chr(1607),  # ﮬ
    chr(64429): chr(1607),  # ﮭ
    chr(65171): chr(1607),  # ﺓ
    chr(65172): chr(1607),  # ﺔ
    chr(65257): chr(1607),  # ﻩ
    chr(65258): chr(1607),  # ﻪ
    chr(65259): chr(1607),  # ﻫ
    chr(65260): chr(1607),  # ﻬ

    # ی
    chr(1609):  chr(1740),  # ى
    chr(1610):  chr(1740),  # ي
    chr(1746):  chr(1740),  # ے
    chr(64484): chr(1740),  # ﯤ
    chr(64485): chr(1740),  # ﯥ
    chr(64486): chr(1740),  # ﯦ
    chr(64487): chr(1740),  # ﯧ
    chr(64508): chr(1740),  # ﯼ
    chr(64509): chr(1740),  # ﯽ
    chr(64510): chr(1740),  # ﯾ
    chr(64511): chr(1740),  # ﯿ
    chr(65263): chr(1740),  # ﻯ
    chr(65264): chr(1740),  # ﻰ
    chr(65265): chr(1740),  # ﻱ
    chr(65266): chr(1740),  # ﻲ
    chr(65267): chr(1740),  # ﻳ
    chr(65268): chr(1740),  # ﻴ

    # ﺉ
    chr(65161): chr(1574),  # ﺉ
    chr(65162): chr(1574),  # ﺊ
    chr(65163): chr(1574),  # ﺋ
    chr(65164): chr(1574),  # ﺌ

    # یی
    chr(1574)+chr(1740):  chr(1740)+chr(1740), # ئی
    chr(1574)+chr(1574):  chr(1740)+chr(1740), # ئئ

    # ه ی
    chr(1620): ' '+chr(1740),
    chr(1728): chr(1607)+' '+chr(1740),

    # Words Forms
    chr(1954):  chr(1576)+chr(1583),                        # ޢ
    chr(65275): chr(1604)+chr(1575),                        # ﻻ
    chr(65276): chr(1604)+chr(1575),                        # ﻼ
    chr(64512): chr(1574)+chr(1580),                        # ﰀ
    chr(64561): chr(1574)+chr(1608),                        # ﯱ
    chr(65008): chr(1589)+chr(1604)+chr(1609),              # ﷰ
    chr(65009): chr(1602)+chr(1604)+chr(1609),              # ﷱ
    chr(65017): chr(1589)+chr(1604)+chr(1609),              # ﷹ
    chr(65020): chr(1585)+chr(1740)+chr(1575)+chr(1604),    # ﷼
    chr(65010): chr(1575)+chr(1604)+chr(1604)+chr(1607),    # ﷲ
    chr(65011): chr(1575)+chr(1603)+chr(1576)+chr(1585),    # ﷳ
    chr(65012): chr(1605)+chr(1581)+chr(1605)+chr(1583),    # ﷴ
    chr(65013): chr(1589)+chr(1604)+chr(1593)+chr(1605),    # ﷵ
    chr(65014): chr(1585)+chr(1587)+chr(1608)+chr(1604),    # ﷶ
    chr(65015): chr(1593)+chr(1604)+chr(1740)+chr(1607),    # ﷷ
    chr(65016): chr(1608)+chr(1587)+chr(1604)+chr(1605),    # ﷸ
    chr(65019): chr(1580)+chr(1604)+chr(1604)+chr(1575)     # ﷻ
                +chr(1584)+chr(1604)+chr(1607),
    chr(65018): (chr(1589)+chr(1604)+chr(1609)              # ﷺ
                +chr(1575)+chr(1604)+chr(1604)+chr(1607)
                +chr(1605)+chr(1593)+chr(1604)+chr(1605)),
    chr(65021): (chr(1576)+chr(1587)+chr(1605)+chr(1600)    #  ﷽
                +chr(1575)+chr(1604)+chr(1604)+chr(1607)
                +chr(1608)+chr(1585)+chr(1575)+chr(1606)
                +chr(1600)+chr(1585)+chr(1575)+chr(1607)
                +chr(1610)+chr(1605)),
    chr(8230): chr(46)+ chr(46)+chr(46),                  # … Horizontal ellipsis
}

ARABIC_NUMBER_MAPPING_DICT = {

    # Normal
    chr(1632): '0',
    chr(1633): '1',
    chr(1634): '2',
    chr(1635): '3',
    chr(1636): '4',
    chr(1637): '5',
    chr(1638): '6',
    chr(1639): '7',
    chr(1640): '8',
    chr(1641): '9',

    # Extended
    chr(1776): '0',
    chr(1777): '1',
    chr(1778): '2',
    chr(1779): '3',
    chr(1780): '4',
    chr(1781): '5',
    chr(1782): '6',
    chr(1783): '7',
    chr(1784): '8',
    chr(1785): '9',

    # Subscript
    chr(8320): '0',
    chr(8321): '1',
    chr(8322): '2',
    chr(8323): '3',
    chr(8324): '4',
    chr(8325): '5',
    chr(8326): '6',
    chr(8327): '7',
    chr(8328): '8',
    chr(8329): '9',

    # Superscript
    chr(8304): '0',
    chr(185):  '1',
    chr(178):  '2',
    chr(179):  '3',
    chr(8308): '4',
    chr(8309): '5',
    chr(8310): '6',
    chr(8311): '7',
    chr(8312): '8',
    chr(8313): '9',
}

PUNCTUATION_MAPPING_DICT = {
    # ? (Question Mark)
    chr(65047): '?',  # Vertical
    chr(65110): '?',  # Small
    chr(65311): '?',  # Fullwidth

    # ! (Exclamation Mark)
    chr(65045): '!',  # Vertical
    chr(65111): '!',  # Small
    chr(65281): '!',  # Fullwidth

    # , (Comma)
    chr(65040): ',',  # Vertical
    chr(65104): ',',  # Small
    chr(65292): ',',  # Fullwidth

    # . (Full Stop)
    chr(1748):  '.',  # Arabic
    chr(65042): '.',  # Vertical
    chr(65106): '.',  # Small
    chr(65294): '.',  # Fullwidth

    # ; (Semicolon)
    chr(65044): ';',  # Vertical
    chr(65108): ';',  # Small
    chr(65307): ';',  # Fullwidth

    # : (Colon)
    chr(65043): ':',  # Vertical
    chr(65109): ':',  # Small
    chr(65306): ':',  # Fullwidth

    # / (Solidus)
    chr(65295): '/',  # Fullwidth

    # \ (Reverse Solidus)
    chr(65128): '\\', # Small
    chr(65340): '\\', # Fullwidth

    # [ (Left Square Bracket)
    chr(65095): '[',  # Vertical
    chr(65115): '[',  # Small
    chr(65339): '[',  # Fullwidth

    # ] (Right Square Bracket)
    chr(65096): ']',  # Vertical
    chr(65116): ']',  # Small
    chr(65341): ']',  # Fullwidth

    # ' (Apostrophe)
    chr(8217):  "'",  # Right Single Quotation Mark
    chr(65287): "'",  # Fullwidth Apostrophe

    # " (Quotation Mark)
    chr(8220):  '"',  # Left Double
    chr(8221):  '"',  # Right Double
    chr(65048): '"',  # Vertical
    chr(65282): '"',  # Fullwidth

    # # (Number Sign)
    chr(65119): '#',  # Small
    chr(65283): '#',  # Fullwidth

    # * (Asterisk)
    chr(65121): '*',  # Small
    chr(65290): '*',  # Fullwidth

    # % (Percentage)
    chr(1642):  '%', # Arabic

    # $ (Dollar Sign)
    chr(65284): '$',  # Fullwidth

    # ^ (Circumflex Accent)
    chr(65342): '^',  # Fullwidth

    # _ (Underscore)
    chr(65343): '_',  # Fullwidth

    # - (Hyphen-Minus)
    chr(173):   '-',  # ­Soft
    chr(65123): '-',  # Small
    chr(65293): '-',  # Fullwidth

    # = (Equals Sign)
    chr(65126): '=',  # Small
    chr(65309): '=',  # Fullwidth

    # + (Plus Sign)
    chr(65122): '+',  # Small
    chr(65291): '+',  # Fullwidth

    # @ (Commercial At)
    chr(65131): '@',  # Small
    chr(65312): '@',  # Fullwidth

    # ( (Left Parenthesis)
    chr(65077): '(',  # Vertical
    chr(65113): '(',  # Small
    chr(65288): '(',  # Fullwidth

    # ) (Right Parenthesis)
    chr(65078): ')',  # Vertical
    chr(65114): ')',  # Small
    chr(65289): ')',  # Fullwidth
}

ARABIC_DIACRITICS_MAPPING  = {
    chr(1761):  chr(1618),       # ۡ  Sukun(Jazm)
    chr(1958):  chr(1614),       # ަ  Fatha
    chr(1959):  chr(1611),       # ާ  Fathatan
    chr(1960):  chr(1616),       # ި  Kasra
    chr(1961):  chr(1613),       # ީ  Kasratan
    chr(1962):  chr(1615),       # ު  Damma
    chr(1963):  chr(1612),       # ޫ  Dammatan
    chr(59424): chr(1614),       #  َ Fatha
    chr(59425): chr(1615),       #  ُ Damma
    chr(59429): chr(1617),       #  ّ Shadda
    chr(59430): chr(1616),       #  ِ Kasra
    chr(65136): chr(1611),       # ﹰ Fathatan isolated form
    chr(65137): chr(1611),       # ﹱ Fathatan medial form
    chr(65138): chr(1613),       # ﹲ Dammatan isolated form
    chr(65140): chr(1615),       # ﹴ Kasratan isolated form
    chr(65142): chr(1617),       # ﹶ Fatha isolated form
    chr(65143): chr(1618),       # ﹷ Fatha medial form
    chr(65144): chr(1615),       # ﹸ Damma isolated form
    chr(65145): chr(1615),       # ﹹ Damma medial form
    chr(65146): chr(1616),       # ﹺ Kasra isolated form
    chr(65147): chr(1613),       # ﹻ Kasra medial form
    chr(65148): chr(1617),       # ﹼ Shadda isolated form
    chr(65149): chr(1617),       # ﹽ Shadda medial form
    chr(65152): chr(1569),       # ﺀ Hamza isolated form
}

REDUNDANT_CHAR_LIST = [
    chr(1600),                  # ـ Tatweel
    chr(8205),                  # Zero-width joiner
    chr(8301),                  # ⁭Activate Arabic form shaping
    chr(8216),                  # ‘ Left single quotation for numbers
]

AI_REDUNDANT_CHAR_LIST = [
    # Arabic Diacritics
    chr(1569),                  #ء Hamza
    chr(1611),                  # ً Fathatan
    chr(1612),                  # ٌ Dammatan
    chr(1613),                  # ٍ Kasratan
    chr(1614),                  # َ Fatha
    chr(1615),                  # ُ Damma
    chr(1616),                  # ِ Kasra
    chr(1617),                  # ّ Shadda
    chr(1618),                  # ْ Sukun
]

AI_SPECIFIC_MAPPING_DICT = {
    # ی
    chr(1574): chr(1740),   # ﺉ

    # ا
    chr(1571): chr(1575),   # أ

    # ?
    chr(1567): '?',        # ؟

}

SPACE_MAPPING_DICT = {ch: ' ' for ch in SPACE_MAPPING_LIST}
REDUDANT_MAPPING_DICT = {ch: '' for ch in REDUNDANT_CHAR_LIST}
AI_REDUDANT_MAPPING_DICT = {ch: '' for ch in AI_REDUNDANT_CHAR_LIST}

###Normalizer

In [ ]:
class PersianTextNormalizer():
    def __init__(self,):
        self.mappings_list = [
            SPACE_MAPPING_DICT,
            ARABIC2PERSIAN_MAPPING_DICT,
            ARABIC_NUMBER_MAPPING_DICT,
            PUNCTUATION_MAPPING_DICT,
            REDUDANT_MAPPING_DICT,
        ]
        self.suffix_ha_exceptions_list = SUFFIX_HA_EXCEPTION_LIST
        self.ha_patterns = re.compile(r"(\w+?)(های|ها)\b")

    def _apply_mapping(self, text: str, mapping: dict) -> str:
        """Replace characters in text according to mapping."""
        for old, new in mapping.items():
            text = text.replace(old, new)
        return text

    def _split_suffix(self, match: re.Match) -> str:
        base, suffix = match.group(1), match.group(2)
        full = base + suffix
        if full in self.suffix_ha_exceptions_list:
            return full
        return f"{base} {suffix}"

    def normalize(self, input: str) -> str:
        try:
            # chain mappings
            text = input
            for mapping in self.mappings_list:
                text = self._apply_mapping(text, mapping)

            # collapse spaces
            text = re.sub(r"\s+", " ", text).strip()
            # split suffix
            text = self.ha_patterns.sub(self._split_suffix, text)
            return text

        except Exception as e:
            print(f"Normalization faced this error: {e}")
            return input

##Tokenizations

In [ ]:
MODEL_ID   = "PartAI/Dorna-Llama3-8B-Instruct"
MODEL_NAME = "Dorna"

train_df = train_df[["text", "labels"]]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, add_prefix_space=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
# Llama-style models often have no pad token; set to EOS for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # keep right padding for training

def tokenize_function(dataset):
    return tokenizer(dataset["text"], truncation=True)

dataset = Dataset.from_pandas(train_df)
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.with_format("numpy")

# Set up the tokenizer pad token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer, padding=True)

##Dataset

In [ ]:

# Your labeled DataFrame must contain:
#   - a user-visible "prompt" (what the model sees)
#   - a gold "response" (the report text you want the model to generate)
# If you currently have columns ["text", "labels"], we’ll map them below.
# Example shape:
# train_df = pd.DataFrame({
#   "text": ["Generate a sales report for ...", ...],
#   "labels": ["Executive Summary...\n1) KPIs ...", ...]
# })

# ---------------------------
# Load / map your data
# ---------------------------
# Replace this with your real DataFrame
# train_df = pd.read_csv("your_file.csv")
# For your given schema:
#   text   -> prompt
#   labels -> response

def build_messages_dataframe(train_df: pd.DataFrame) -> pd.DataFrame:
    df = train_df[["text", "labels"]].rename(columns={"text": "prompt", "labels": "response"}).copy()

    # Optional: a fixed system style guide to push the model toward advanced sales reporting
    SYSTEM_PROMPT = (
        "You are a senior financial analyst. Write advanced, structured sales reports for companies. "
        "Use clear headings (Executive Summary, KPIs, Trend Analysis, Variances, Risks, Recommendations), "
        "be precise and actionable, and keep a professional tone."
    )

    # Convert each row into chat 'messages' compatible with Llama-3 chat template
    def to_messages(row):
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": row["response"]},
        ]

    df["messages"] = df.apply(to_messages, axis=1)
    return df[["messages"]]



# (Replace this with your real train_df)
train_df = pd.DataFrame({
    "text": [
        # PROMPT EXAMPLE (User instruction): include structured/JSON context or free text —
        # this is what your RAG/system would give the model at inference time.
        "Using the provided monthly sales metrics for ACME Co. (FY2023 Q4 vs Q3), write a comprehensive sales report. "
        "Focus on revenue growth, margins, regional performance, top SKUs, anomalies, and concrete recommendations."
    ],
    "labels": [
        # RESPONSE EXAMPLE (Gold answer you want the model to learn to produce)
        "Executive Summary:\n• Revenue grew 8.4% QoQ, led by EMEA (+13%).\n\nKPIs:\n• Revenue: $12.4M (+8.4%)\n"
        "• Gross Margin: 38.9% (+120 bps)\n• CAC Payback: 9.1 months (vs 9.7)\n\nTrend Analysis:\n• Seasonal uplift..."
    ]
})

df_messages = build_messages_dataframe(train_df)
raw_ds = Dataset.from_pandas(df_messages)

# Keep a small validation split (10%)
ds = raw_ds.train_test_split(test_size=0.1, seed=42)
train_ds = ds["train"]
eval_ds  = ds["test"]

# We format messages into a single training string using the model’s chat template.
def chat_formatting_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False  # include the gold assistant in the text
        )
        texts.append(text)
    return {"text": texts}

train_ds = train_ds.map(chat_formatting_func, batched=True, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(chat_formatting_func,  batched=True, remove_columns=eval_ds.column_names)

# ---------------------------
# Data collator to label ONLY the assistant part
# ---------------------------
# Build the "assistant header" template so the collator knows where labels begin
assistant_template = tokenizer.apply_chat_template(
    [{"role": "assistant", "content": ""}],
    tokenize=False,
    add_generation_prompt=False
)
collator = DataCollatorForCompletionOnlyLM(
    response_template=assistant_template,
    tokenizer=tokenizer
)

#LLM Model

##Configs

In [ ]:
# Quantization configs
FOUR_BIT_MODE = True
FOUR_BIT_Q_TYPE = 'nf4' # or "fp4"
FOUR_BIT_DOUBLEQ = True # nested quantization

EIGHT_BIT_MODE = False
EIGHT_BIT_THRESHOLD = 6.0
EIGHT_BIT_CPU_OFFLOAD = False

# LoRA (PEFT) configs
LR_DIM = 16
LORA_ALPHA = 8
LORA_DROPOUT = 0.05
TASK_TYPE = 'CAUSAL_LM' # or SEQ_CLS
LORA_Target_Modules_List = [
    'q_proj',    # Query Projection Layer (Most Impactful)
    'v_proj',    # Value Projection Layer (Most Impactful)
    'k_proj',    # Key Projection Layer
    'o_proj',    # Output Projection Layer
    "gate_proj", # Gate Projection Layer
    "up_proj",   # Up Projection Layer
    "down_proj"  # Down Projection Layer

]
# Set target_modules=None to let PEFT infer names based on model_type

qb_kwargs = {
    "load_in_4bit": FOUR_BIT_MODE,
    "bnb_4bit_quant_type": FOUR_BIT_Q_TYPE,
    "bnb_4bit_use_double_quant": FOUR_BIT_DOUBLEQ,
    "bnb_4bit_compute_dtype": torch.bfloat16,
}

if EIGHT_BIT_MODE:
    qb_kwargs.update({
        "load_in_8bit": True,
        "llm_int8_threshold": EIGHT_BIT_THRESHOLD,
        "llm_int8_skip_modules": ["lm_head"],
        "llm_int8_enable_fp32_cpu_offload": EIGHT_BIT_CPU_OFFLOAD,
    })

config = AutoConfig.from_pretrained(MODEL_ID)

quantization_config = BitsAndBytesConfig(**qb_kwargs)

lora_config = LoraConfig(
    r = LR_DIM,
    lora_alpha = LORA_ALPHA,
    target_modules = LORA_Target_Modules_List,
    lora_dropout = LORA_DROPOUT,
    bias = 'none',
    task_type = 'CAUSAL_LM',
)


##Model Initialization

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map={"": 0},
)

##PEFT Model

In [ ]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.pretraining_tp = 1

#Training Model

In [ ]:
EPOCHS = 4
LEARNING_RATE = 1e-6
DEVICE_BATCH_SIZE = 2

EFF_BATCH_SIZE = 64
EVAL_STEPS = 200
LOGGING_STEP = 50
WARM_UP_RATIO = 0.3
WEIGHT_DECAY = 0
MAX_NORM_GRAD = 1
MAX_SEQ_LENGTH = 4096

WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
grad_acc_steps = max(1, EFF_BATCH_SIZE // (DEVICE_BATCH_SIZE * WORLD_SIZE))

RUN_NAME = f"{MODEL_NAME}_T_{EPOCHS}_L_{LEARNING_RATE}_B_{DEVICE_BATCH_SIZE}"
MODEL_PATH = f"./{MODEL_NAME}_LLM"

training_args = TrainingArguments(
    output_dir="Dorna",
    run_name=RUN_NAME,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,                         # typical for QLoRA SFT (higher than full-fp16)
    per_device_train_batch_size=DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=grad_acc_steps,
    logging_steps=LOGGING_STEP,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="epoch",
    lr_scheduler_type="cosine",
    warmup_ratio=WARM_UP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_NORM_GRAD,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=not (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8),
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,       # adjust if your GPU allows; check the model’s context length
    packing=False,             # True packs examples; keep False unless you know you want packing
)

train_result = trainer.train()

#Saving Model

In [ ]:
trainer.train()
trainer.save_model("dorna_out/adapter")  # saves LoRA adapters
tokenizer.save_pretrained("dorna_out/tokenizer")

saved_model = trainer.model
saved_model = saved_model.merge_and_unload()
saved_model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

#Evaluating Model

##Model Evaluation

In [ ]:
gen = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, device=0)
print(gen("شروع متن شما...", max_length=100))